In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import time
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime
import re
import os

options = Options()
#options.add_argument("--headless")
options.add_argument("--disable-gpu")
options.add_argument("--window-size=1920,1080")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36")

def normalize_percent_str(s: str) -> str:
    if not s:
        return ""
    s = str(s).strip()
    m = re.search(r'(-?\d+(?:[\.,]\d+)?)\s*%', s)
    if not m:
        return ""
    val = m.group(1).replace(",", ".")
    try:
        num = abs(float(val))
    except:
        return ""
    if num.is_integer():
        return f"{int(num)}%"
    return f"{num}%"

def extraer_info_producto_elektra(product, fecha_consulta):
    try:
        nombre = ""
        nombre_tag = product.find("span", class_="vtex-product-summary-2-x-productBrand")
        if nombre_tag:
            nombre = nombre_tag.get_text(strip=True)

        marca = ""
        if nombre:
            palabras = nombre.split()
            if palabras:
                marca = palabras[0].upper()
                if marca == "IPHONE":
                    marca = "APPLE"

        color = ""
        if nombre:
            nombre_lower = nombre.lower()
            colores = [
                'titanio negro', 'titanio blanco', 'titanio del desierto', 'titanio natural', 'negro',
                'black', 'blanco', 'white', 'azul', 'blue', 'rojo', 'red',
                'verde', 'green', 'amarillo', 'yellow', 'rosa', 'pink', 'morado', 'purple', 'purpura',
                'gris', 'gray', 'grey', 'dorado', 'gold', 'plateado', 'silver', 'plata', 'cian',
                'lila', 'grafito', 'lavanda', 'ultramarino', 'medianoche', 'cafe',
                'menta', 'multicolor', 'beige', 'violeta', 'naranja', 'midnight', 'titanio', 'starlight'
            ]
            for c in colores:
                if c in nombre_lower:
                    color = c.capitalize()
                    break

        capacidad = ""
        if nombre:
            capacidad_match = re.search(r'(\d+(?:\.\d+)?)\s*(GB|TB)', nombre, re.IGNORECASE)
            if capacidad_match:
                numero = capacidad_match.group(1)
                unidad = capacidad_match.group(2).upper()
                capacidad = f"{numero} {unidad}"

        url_producto = ""
        if hasattr(product, 'name') and product.name == 'a':
            if product.get("href"):
                href = product["href"]
                if href.startswith("/"):
                    url_producto = "https://www.elektra.mx" + href
                elif href.startswith("https://"):
                    url_producto = href
        else:
            link_tag = product.find("a", class_="vtex-product-summary-2-x-clearLink")
            if not link_tag:
                link_tag = product.find("a", href=True)
            if link_tag and link_tag.get("href"):
                href = link_tag["href"]
                if href.startswith("/"):
                    url_producto = "https://www.elektra.mx" + href
                elif href.startswith("https://"):
                    url_producto = href

        sku = ""
        cite_tag = product.find("cite", {"data-id": True})
        if cite_tag:
            sku = cite_tag.get("data-id", "")
        if not sku and url_producto:
            sku_match = re.search(r'-(\d{8})/p$', url_producto)
            if sku_match:
                sku = sku_match.group(1)
            else:
                sku_match = re.search(r'/([^/]*-)?(\d{6,})/p$', url_producto)
                if sku_match:
                    sku = sku_match.group(2)

        precio_original = ""
        precio_descuento = ""
        porcentaje_descuento = ""

        def _first_text(node):
            if not node:
                return ""
            return node.get_text(strip=True)

        # --- CONTENEDOR DE PRECIOS ACTUAL ---
        # Sin descuento:  <div class="containerPricePlp"><div class="priceNoList">$X</div></div>
        # Con descuento:  <div class="listPrice">$X</div>
        #                 <div class="containerPricePlp">
        #                   <div class="price">$Y</div>
        #                   <div class="descuento">-Z%</div>
        #                 </div>
        list_price_new    = product.select_one("div.listPrice")
        selling_price_new = product.select_one("div.containerPricePlp div.price")
        no_list_price_new = product.select_one("div.containerPricePlp div.priceNoList")
        descuento_new     = product.select_one("div.containerPricePlp div.descuento")

        if no_list_price_new and not list_price_new:
            # Sin descuento
            raw = _first_text(no_list_price_new)
            if raw:
                precio_original = raw
                precio_descuento = "No discount"
                porcentaje_descuento = ""
        elif list_price_new or selling_price_new:
            # Con descuento
            precio_original = _first_text(list_price_new)
            precio_descuento = _first_text(selling_price_new)
            porcentaje_descuento = normalize_percent_str(_first_text(descuento_new))

        # --- SELECTORES ANTERIORES (fallback) ---
        if not precio_original and not precio_descuento:
            precio_orig_tag = product.find("span", class_="vtex-store-components-3-x-listPriceValue")
            if precio_orig_tag:
                precio_span = precio_orig_tag.find("span", class_="vtex-store-components-3-x-currencyContainer")
                if precio_span:
                    precio_inner_span = precio_span.find("span")
                    if precio_inner_span:
                        precio_original = precio_inner_span.get_text(strip=True)

            precio_desc_tag = product.find("span", class_="vtex-store-components-3-x-sellingPriceValue")
            if precio_desc_tag:
                precio_span = precio_desc_tag.find("span")
                if precio_span:
                    precio_descuento = precio_span.get_text(strip=True)

            if not precio_descuento:
                precio_desc_div = product.find("div", class_="vtex-store-components-3-x-sellingPrice")
                if precio_desc_div:
                    precio_span = precio_desc_div.find("span", class_="vtex-store-components-3-x-currencyContainer")
                    if precio_span:
                        precio_inner_span = precio_span.find("span")
                        if precio_inner_span:
                            precio_descuento = precio_inner_span.get_text(strip=True)

            if not precio_descuento:
                precio_desc_tag = product.find("span", class_="vtex-store-components-3-x-currencyContainer")
                if precio_desc_tag:
                    precio_span = precio_desc_tag.find("span")
                    if precio_span:
                        precio_descuento = precio_span.get_text(strip=True)

        if precio_original and precio_descuento:
            pass
        elif not precio_original and precio_descuento:
            precio_original = precio_descuento
            precio_descuento = "No discount"
        elif precio_original and not precio_descuento:
            precio_descuento = "No discount"
        else:
            precio_original = "No inventory"
            precio_descuento = "No discount"

        def limpiar_precio(precio):
            if precio and precio not in ["No discount", "No inventory"]:
                cleaned = re.sub(r'[^\d.]', '', precio.replace(',', ''))
                try:
                    return float(cleaned)
                except:
                    return None
            return None

        p_orig = limpiar_precio(precio_original)
        p_desc = limpiar_precio(precio_descuento)

        if porcentaje_descuento:
            porcentaje_str = porcentaje_descuento
        elif p_orig and p_desc and p_orig > p_desc:
            porcentaje_calc = abs(round(100 * (p_orig - p_desc) / p_orig, 2))
            porcentaje_str = f"{porcentaje_calc}%"
        else:
            porcentaje_str = "No discount"

        return {
            "Marca": marca,
            "Nombre": nombre,
            "Color": color,
            "Capacidad": capacidad,
            "SKU": sku,
            "URL": url_producto,
            "FechaConsulta": fecha_consulta,
            "Precio_original": precio_original,
            "Precio_descuento": precio_descuento,
            "Porcentaje_descuento": porcentaje_str
        }

    except Exception as e:
        print(f"Error al extraer producto: {e}")
        return None

def scrape_elektra_smartphones():
    url = "https://www.elektra.mx/telefonia/celulares/smartphones?initialMap=c,c,c&initialQuery=telefonia/celulares/smartphones&map=category-1,category-2,category-3,sellername&query=/telefonia/celulares/smartphones/elektra&searchState"

    print("Iniciando scraping de Elektra...")

    driver = webdriver.Chrome(options=options)
    driver.get(url)
    time.sleep(10)

    data = []
    fecha_consulta = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    try:
        max_clicks = 100
        clicks_realizados = 0

        while clicks_realizados < max_clicks:
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(5)
            try:
                mostrar_mas_btn = driver.find_element(By.CSS_SELECTOR, "div.elektra-search-result-3-x-buttonShowMore button")
                if mostrar_mas_btn.is_displayed() and mostrar_mas_btn.is_enabled():
                    print(f"Haciendo clic en 'Mostrar mas' (clic #{clicks_realizados + 1})")
                    driver.execute_script("arguments[0].scrollIntoView(true);", mostrar_mas_btn)
                    time.sleep(1)
                    driver.execute_script("arguments[0].click();", mostrar_mas_btn)
                    clicks_realizados += 1
                    time.sleep(15)
                else:
                    print("Boton 'Mostrar mas' no disponible")
                    break
            except Exception:
                print("No se encontro el boton 'Mostrar mas' - todos los productos cargados")
                break

        print(f"Total de clics en 'Mostrar mas': {clicks_realizados}")
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(3)

        soup = BeautifulSoup(driver.page_source, "html.parser")

        products = soup.find_all("a", class_="vtex-product-summary-2-x-clearLink")
        print(f"Estrategia 1 - Enlaces encontrados: {len(products)}")

        if not products:
            products = soup.find_all("div", class_="vtex-search-result-3-x-galleryItem")
            print(f"Estrategia 2 - galleryItem: {len(products)}")

        if not products:
            products = soup.find_all("div", class_="vtex-product-summary-2-x-container")
            print(f"Estrategia 3 - container: {len(products)}")

        if not products:
            products = soup.find_all("article", class_="vtex-product-summary-2-x-element")
            print(f"Estrategia 4 - article: {len(products)}")

        print(f"Productos encontrados: {len(products)}")

        for i, product in enumerate(products[:3]):
            print(f"\n--- Debug Producto {i+1} ---")
            lp  = product.select_one("div.listPrice")
            sp  = product.select_one("div.containerPricePlp div.price")
            nlp = product.select_one("div.containerPricePlp div.priceNoList")
            dc  = product.select_one("div.containerPricePlp div.descuento")
            print(f"listPrice: {lp.get_text(strip=True) if lp else None}")
            print(f"price:     {sp.get_text(strip=True) if sp else None}")
            print(f"priceNoList: {nlp.get_text(strip=True) if nlp else None}")
            print(f"descuento: {dc.get_text(strip=True) if dc else None}")

        for product in products:
            info = extraer_info_producto_elektra(product, fecha_consulta)
            if info and (info["Nombre"] or info["URL"]):
                data.append(info)
                print(f"Extraido: {info['Nombre'][:40] if info['Nombre'] else 'Sin nombre'} | {info['Precio_original']} | {info['Precio_descuento']}")

        print(f"Productos validos extraidos: {len(data)}")

    except Exception as e:
        print(f"Error durante el scraping: {e}")
        import traceback
        traceback.print_exc()
    finally:
        driver.quit()

    return data

resultados_elektra = scrape_elektra_smartphones()

if resultados_elektra:
    df_elektra = pd.DataFrame(resultados_elektra)
    fecha_hoy = datetime.now().strftime("%Y-%m-%d")
    nombre_archivo = f"elektra_smartphones_{fecha_hoy}.csv"
    try:
        df_elektra.to_csv(nombre_archivo, index=False, encoding="utf-8-sig")
    except PermissionError:
        sufijo = datetime.now().strftime("%H%M%S")
        nombre_archivo = f"elektra_smartphones_{fecha_hoy}_{sufijo}.csv"
        df_elektra.to_csv(nombre_archivo, index=False, encoding="utf-8-sig")
    print(f"Datos exportados a {nombre_archivo}")
    print(f"Total de productos: {len(df_elektra)}")
    display(df_elektra.head())
else:
    print("No se encontraron productos.")
